# 02 Extract WLASL1000 Keypoints

## Purpose
This notebook converts WLASL1000 videos into MediaPipe keypoint arrays.

Each output file has shape:

```text
(60, 258)
```

## Extracted features per frame
- Left hand: 21 landmarks × 3 = 63
- Right hand: 21 landmarks × 3 = 63
- Pose: 33 landmarks × 4 = 132
- Total: 258 features

In [ ]:
from pathlib import Path
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

In [ ]:
PROJECT_ROOT = Path("E:/Be_My_Ear")
DATASET_NAME = "WLASL1000"
PREFIX = "wlasl1000"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ASL" / DATASET_NAME
KEYPOINT_DIR = PROCESSED_DIR / "keypoints"
KEYPOINT_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_INDEX_FILE = PROCESSED_DIR / f"{PREFIX}_video_index.csv"
KEYPOINT_INDEX_FILE = PROCESSED_DIR / f"{PREFIX}_keypoint_index.csv"
FAILED_LOG_FILE = PROCESSED_DIR / f"{PREFIX}_failed_keypoint_extraction.csv"

df = pd.read_csv(VIDEO_INDEX_FILE)

print("Videos to process:", len(df))
print("Classes:", df["label_id"].nunique())
print("Keypoint output folder:", KEYPOINT_DIR)

## 1. Test video reading

In [ ]:
test_video = df.iloc[0]["video_path"]
cap = cv2.VideoCapture(test_video)

print("Testing video:", test_video)
print("Opened:", cap.isOpened())
print("Frame count:", int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
print("FPS:", cap.get(cv2.CAP_PROP_FPS))

cap.release()

## 2. Set up MediaPipe and extraction functions

In [ ]:
print("MediaPipe version:", getattr(mp, "__version__", "No version found"))
print("Has mp.solutions:", hasattr(mp, "solutions"))

mp_holistic = mp.solutions.holistic

SEQUENCE_LENGTH = 60
LEFT_HAND_SIZE = 21 * 3
RIGHT_HAND_SIZE = 21 * 3
POSE_SIZE = 33 * 4
FEATURE_SIZE = LEFT_HAND_SIZE + RIGHT_HAND_SIZE + POSE_SIZE

print("Feature size per frame:", FEATURE_SIZE)


def extract_landmarks_from_results(results):
    if results.left_hand_landmarks:
        left_hand = np.array([[lm.x, lm.y, lm.z] for lm in results.left_hand_landmarks.landmark], dtype=np.float32).flatten()
    else:
        left_hand = np.zeros(LEFT_HAND_SIZE, dtype=np.float32)

    if results.right_hand_landmarks:
        right_hand = np.array([[lm.x, lm.y, lm.z] for lm in results.right_hand_landmarks.landmark], dtype=np.float32).flatten()
    else:
        right_hand = np.zeros(RIGHT_HAND_SIZE, dtype=np.float32)

    if results.pose_landmarks:
        pose = np.array([[lm.x, lm.y, lm.z, lm.visibility] for lm in results.pose_landmarks.landmark], dtype=np.float32).flatten()
    else:
        pose = np.zeros(POSE_SIZE, dtype=np.float32)

    return np.concatenate([left_hand, right_hand, pose])


def extract_keypoints_from_video(video_path, sequence_length=60):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        return None

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if frame_count <= 0:
        cap.release()
        return None

    frame_indices = np.linspace(0, frame_count - 1, sequence_length).astype(int)
    sequence = []

    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        enable_segmentation=False,
        refine_face_landmarks=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:

        for frame_idx in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
            success, frame = cap.read()

            if not success:
                sequence.append(np.zeros(FEATURE_SIZE, dtype=np.float32))
                continue

            image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image_rgb.flags.writeable = False

            results = holistic.process(image_rgb)
            sequence.append(extract_landmarks_from_results(results))

    cap.release()
    return np.array(sequence, dtype=np.float32)

## 3. Test extraction on one video

In [ ]:
sample_row = df.iloc[0]
sample = extract_keypoints_from_video(sample_row["video_path"], SEQUENCE_LENGTH)

print("Sample video:", sample_row["video_path"])
print("Sample shape:", None if sample is None else sample.shape)

## 4. Test first 10 videos

In [ ]:
test_df = df.head(10).copy()

success_count = 0
fail_count = 0

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Testing first 10 videos"):
    output_path = KEYPOINT_DIR / f"{row['video_id']}.npy"
    keypoints = extract_keypoints_from_video(row["video_path"], SEQUENCE_LENGTH)

    if keypoints is None:
        fail_count += 1
        continue

    np.save(output_path, keypoints)
    success_count += 1

print("Test extraction success:", success_count)
print("Test extraction failed:", fail_count)

## 5. Full WLASL1000 extraction

This step can take a long time. It is resumable because existing `.npy` files are skipped.

In [ ]:
success_count = 0
fail_count = 0
skipped_count = 0
failed_records = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting WLASL1000 keypoints"):
    output_path = KEYPOINT_DIR / f"{row['video_id']}.npy"

    if output_path.exists():
        skipped_count += 1
        continue

    keypoints = extract_keypoints_from_video(row["video_path"], SEQUENCE_LENGTH)

    if keypoints is None:
        fail_count += 1
        failed_records.append({
            "video_id": row["video_id"],
            "video_path": row["video_path"],
            "reason": "video_open_or_frame_failed"
        })
        continue

    np.save(output_path, keypoints)
    success_count += 1

print("Extraction completed")
print("Success:", success_count)
print("Skipped:", skipped_count)
print("Failed:", fail_count)

## 6. Save keypoint index

In [ ]:
pd.DataFrame(failed_records).to_csv(FAILED_LOG_FILE, index=False)

records = []

for _, row in df.iterrows():
    keypoint_path = KEYPOINT_DIR / f"{row['video_id']}.npy"

    if keypoint_path.exists():
        records.append({
            "video_id": row["video_id"],
            "gloss": row["gloss"],
            "label_id": int(row["label_id"]),
            "original_class_id": int(row["original_class_id"]),
            "video_path": row["video_path"],
            "keypoint_path": str(keypoint_path)
        })

keypoint_df = pd.DataFrame(records)
keypoint_df.to_csv(KEYPOINT_INDEX_FILE, index=False)

print("Saved failure log:", FAILED_LOG_FILE)
print("Saved keypoint index:", KEYPOINT_INDEX_FILE)
print("Keypoint samples:", len(keypoint_df))
print("Classes:", keypoint_df["label_id"].nunique())

keypoint_df.head()